# NBA Shot Quality: Temporal Validation (Train 2023 → Test 2024)

This notebook performs **temporal validation** by training models on 2022-23 season data and testing on 2024-25 season data.

**Key Difference from Main Notebook:**
- Main notebook: Random 80/20 split within one season
- This notebook: Train on entire 2023 season, test on entire 2024 season
- **Why**: Validates that model generalizes across seasons (true out-of-sample test)

**Data Required:**
- `enriched_data/nbastatsv3_2023_enriched_shots.csv` (training)
- `enriched_data/nbastatsv3_2024_enriched_shots.csv` (testing)

## Pipeline
1. Load 2023 (train) and 2024 (test) data
2. Feature engineering (consistent features across both seasons)
3. Train models on 2023 data only
4. Evaluate on 2024 data (true temporal holdout)
5. Compare results to within-season validation
6. Export metrics and predictions

In [ ]:
# --- Imports & config
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

TRAIN_PATH = 'enriched_data/nbastatsv3_2023_enriched_shots.csv'
TEST_PATH = 'enriched_data/nbastatsv3_2024_enriched_shots.csv'
RANDOM_STATE = 42

print("Configuration:")
print(f"  Train: {TRAIN_PATH}")
print(f"  Test:  {TEST_PATH}")
print(f"  Random State: {RANDOM_STATE}")

## 1) Load Train (2023) and Test (2024) Data

In [ ]:
# Check files exist
if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(f"Training data not found: {TRAIN_PATH}")
if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(f"Test data not found: {TEST_PATH}")

# Load both datasets
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

print("=" * 80)
print("DATA LOADED")
print("=" * 80)
print(f"Train (2023): {len(df_train):,} shots")
print(f"Test (2024):  {len(df_test):,} shots")
print(f"Total:        {len(df_train) + len(df_test):,} shots")
print("=" * 80)

# Preview
print("\nTrain data preview:")
display(df_train.head(3))
print("\nTest data preview:")
display(df_test.head(3))

## 2) Feature Engineering

Create target variable and feature sets for both train and test.
**Critical**: Use same features for both seasons.

In [ ]:
def prepare_features(df, dataset_name="data"):
    """
    Prepare features and target for a dataset.
    Returns: X (features), y (target), df (cleaned dataframe)
    """
    print(f"\nPreparing {dataset_name}...")
    
    # --- Target creation (case-insensitive)
    col_map = {c.lower(): c for c in df.columns}
    
    if 'shotresult' in col_map:
        col = col_map['shotresult']
        y = df[col].astype(str).str.strip().str.lower().map({'made':1, 'missed':0})
    elif 'isfieldgoal' in col_map:
        col = col_map['isfieldgoal']
        y = pd.to_numeric(df[col], errors='coerce')
    else:
        raise ValueError(f"{dataset_name}: Neither 'shotResult' nor 'isFieldGoal' found")
    
    # Drop unlabeled shots
    mask = ~y.isna()
    dropped = (~mask).sum()
    y = y[mask].astype(int)
    df = df.loc[mask].reset_index(drop=True)
    print(f"  Labeled shots: {len(y):,} | Dropped: {dropped:,}")
    print(f"  Made: {y.sum():,} ({y.mean()*100:.1f}%) | Missed: {(~y.astype(bool)).sum():,} ({(1-y.mean())*100:.1f}%)")
    
    # Standardize text columns
    for c in ['actionType','subType','contest_label','teamTricode','location','description']:
        if c in df.columns:
            df[c] = df[c].astype(str)
    
    # Define features
    num_features = [c for c in [
        'shotDistance','SHOT_CLOCK_APPROX','xLegacy','yLegacy','period',
        'contest_score','shotValue','ABS_TIME'
    ] if c in df.columns]
    
    # === CRITICAL: EXCLUDE LEAKAGE COLUMNS ===
    LEAKAGE_COLUMNS = ['actionType', 'description', 'shotResult', 'isFieldGoal']
    
    cat_features = [c for c in [
        'subType','contest_label','teamTricode','location'
    ] if c in df.columns and c not in LEAKAGE_COLUMNS]
    
    X = df[num_features + cat_features].copy()
    
    print(f"  Numeric features ({len(num_features)}): {num_features}")
    print(f"  Categorical features ({len(cat_features)}): {cat_features}")
    print(f"  Feature matrix shape: {X.shape}")
    
    return X, y, df, num_features, cat_features

# Prepare both datasets
print("=" * 80)
print("FEATURE ENGINEERING")
print("=" * 80)

X_train, y_train, df_train, num_features, cat_features = prepare_features(df_train, "Train (2023)")
X_test, y_test, df_test, num_features_test, cat_features_test = prepare_features(df_test, "Test (2024)")

# Verify feature consistency
print("\n" + "=" * 80)
print("FEATURE CONSISTENCY CHECK")
print("=" * 80)
if num_features == num_features_test and cat_features == cat_features_test:
    print("✅ Features are consistent across train and test")
else:
    print("❌ WARNING: Feature mismatch between train and test!")
    print(f"  Train numeric: {num_features}")
    print(f"  Test numeric:  {num_features_test}")
    print(f"  Train categorical: {cat_features}")
    print(f"  Test categorical:  {cat_features_test}")
print("=" * 80)

## 3) Preprocessing Pipelines

Create sparse (linear models) and dense (tree models) preprocessing pipelines.

In [ ]:
# Sparse branch (linear/kNN/MLP)
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])
transformers = []
if num_features: transformers.append(("num", numeric_transformer, num_features))
if cat_features: transformers.append(("cat", categorical_transformer, cat_features))
preprocess = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0.3)

# Dense branch (trees/boosters)
numeric_transformer_dense = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer_dense = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
transformers_dense = []
if num_features: transformers_dense.append(("num", numeric_transformer_dense, num_features))
if cat_features: transformers_dense.append(("cat", categorical_transformer_dense, cat_features))
preprocess_dense = ColumnTransformer(transformers=transformers_dense, remainder="drop", sparse_threshold=0.0)

print("Preprocessing pipelines created:")
print(f"  Sparse:  {len(transformers)} transformers")
print(f"  Dense:   {len(transformers_dense)} transformers")

## 4) Model Training (2023) and Evaluation (2024)

Train all models on 2023 season, evaluate on 2024 season.

In [ ]:
# Try imports
try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except Exception:
    CATBOOST_AVAILABLE = False
    print("CatBoost not available")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available")

# Define models
models = [
    ("log_reg", LogisticRegression(solver="saga", max_iter=1000, random_state=RANDOM_STATE), False),
    ("knn", KNeighborsClassifier(n_neighbors=25), False),
    ("rf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1), True),
    ("gb", GradientBoostingClassifier(random_state=RANDOM_STATE), True),
    ("mlp", MLPClassifier(hidden_layer_sizes=(64,), max_iter=200, random_state=RANDOM_STATE), False),
]

if CATBOOST_AVAILABLE:
    models.append((
        "catboost",
        CatBoostClassifier(
            depth=6,
            learning_rate=0.1,
            iterations=500,
            loss_function="Logloss",
            eval_metric="AUC",
            random_state=RANDOM_STATE,
            verbose=False
        ),
        True
    ))

if XGBOOST_AVAILABLE:
    models.append((
        "xgboost",
        XGBClassifier(
            n_estimators=600,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        True
    ))

def make_pipeline_for(name, clf, use_dense):
    return Pipeline(steps=[
        ("preprocess", preprocess_dense if use_dense else preprocess),
        ("clf", clf)
    ])

print(f"\nTraining {len(models)} models on 2023 data...")
print("=" * 80)

In [ ]:
results = []
fitted = {}

for name, clf, use_dense in models:
    print(f"\nTraining {name}...")
    pipe = make_pipeline_for(name, clf, use_dense)
    
    # Train on 2023
    pipe.fit(X_train, y_train)
    
    # Evaluate on 2024
    y_proba = pipe.predict_proba(X_test)[:,1]
    y_pred = (y_proba >= 0.5).astype(int)
    
    res = {
        "model": name,
        "test_roc_auc": roc_auc_score(y_test, y_proba),
        "test_log_loss": log_loss(y_test, y_proba, labels=[0,1]),
        "test_brier": brier_score_loss(y_test, y_proba),
        "test_accuracy": accuracy_score(y_test, y_pred)
    }
    
    print(f"  AUC: {res['test_roc_auc']:.4f} | Accuracy: {res['test_accuracy']*100:.1f}% | Log Loss: {res['test_log_loss']:.4f}")
    
    results.append(res)
    fitted[name] = pipe

results_df = pd.DataFrame(results).sort_values("test_roc_auc", ascending=False)

print("\n" + "=" * 80)
print("TEMPORAL VALIDATION RESULTS (Train: 2023 → Test: 2024)")
print("=" * 80)
display(results_df)
print("=" * 80)

## 5) Comparison: Temporal vs Within-Season Validation

Compare these results to the within-season 80/20 split results.

In [ ]:
print("=" * 80)
print("VALIDATION COMPARISON")
print("=" * 80)

print("\n1. TEMPORAL VALIDATION (This Notebook):")
print("   Train: 2022-23 season (100% of 2023 data)")
print("   Test:  2024-25 season (100% of 2024 data)")
print(f"   Train size: {len(X_train):,} shots")
print(f"   Test size:  {len(X_test):,} shots")
print("   Advantage: True out-of-sample (different season)")
print("   Disadvantage: Smaller training set, potential season-to-season drift")

print("\n2. WITHIN-SEASON VALIDATION (Main Notebook):")
print("   Train: 80% of 2024-25 season")
print("   Test:  20% of 2024-25 season")
print("   Train size: ~175k shots")
print("   Test size:  ~44k shots")
print("   Advantage: Larger training set, same season (no drift)")
print("   Disadvantage: Train/test from same distribution")

print("\n" + "=" * 80)
print("KEY INSIGHTS:")
print("=" * 80)

best_model = results_df.iloc[0]
print(f"\nBest Model: {best_model['model']}")
print(f"  Temporal Validation Accuracy: {best_model['test_accuracy']*100:.1f}%")
print(f"  Temporal Validation AUC: {best_model['test_roc_auc']:.4f}")

print("\nExpected comparison:")
print("  - If temporal accuracy ≈ within-season accuracy: Model generalizes well across seasons")
print("  - If temporal accuracy < within-season accuracy: Some season-specific overfitting")
print("  - Typical drop: 1-3 percentage points is normal for temporal validation")

print("\n" + "=" * 80)

## 6) Export Results

In [ ]:
# Save results
output_file = 'model_results_temporal_2023_2024.csv'
results_df.to_csv(output_file, index=False)
print(f"Results saved to: {output_file}")

# Create comparison summary
summary = {
    'validation_type': 'Temporal (2023→2024)',
    'train_season': '2022-23',
    'test_season': '2024-25',
    'train_size': len(X_train),
    'test_size': len(X_test),
    'best_model': best_model['model'],
    'best_auc': best_model['test_roc_auc'],
    'best_accuracy': best_model['test_accuracy'],
    'best_log_loss': best_model['test_log_loss']
}

summary_df = pd.DataFrame([summary])
summary_file = 'temporal_validation_summary.csv'
summary_df.to_csv(summary_file, index=False)
print(f"Summary saved to: {summary_file}")

print("\n" + "=" * 80)
print("TEMPORAL VALIDATION COMPLETE")
print("=" * 80)
print(f"\nFiles created:")
print(f"  - {output_file}")
print(f"  - {summary_file}")
print("\nNext steps:")
print("  1. Compare temporal results to within-season results")
print("  2. Include both validations in your paper")
print("  3. Use temporal validation to demonstrate generalization")
print("=" * 80)

## 7) Visualization: Model Comparison

In [ ]:
# Bar chart of accuracies
plt.figure(figsize=(10, 6))
plt.barh(results_df['model'], results_df['test_accuracy'] * 100)
plt.xlabel('Accuracy (%)', fontsize=12)
plt.ylabel('Model', fontsize=12)
plt.title('Temporal Validation: Train 2023 → Test 2024', fontsize=14, fontweight='bold')
plt.axvline(53.3, color='red', linestyle='--', label='Baseline (always predict miss)', alpha=0.7)
plt.legend()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# AUC comparison
plt.figure(figsize=(10, 6))
plt.barh(results_df['model'], results_df['test_roc_auc'])
plt.xlabel('ROC AUC', fontsize=12)
plt.ylabel('Model', fontsize=12)
plt.title('Temporal Validation: ROC AUC (Train 2023 → Test 2024)', fontsize=14, fontweight='bold')
plt.axvline(0.5, color='red', linestyle='--', label='Random', alpha=0.7)
plt.legend()
plt.grid(axis='x', alpha=0.3)
plt.xlim(0, 1)
plt.tight_layout()
plt.show()